# PkVision — Train on Google Colab

Fine-tune VideoMAE (Kinetics-400) on your parkour clips with GPU acceleration.

**Setup:** Runtime > Change runtime type > **T4 GPU** (or A100 with Pro)

**Steps:**
1. Clone repo + upload training data
2. Fine-tune classifier head (~2 min on T4, ~30s on A100)
3. Plot training curves
4. Test on a video
5. Download trained model

In [ ]:
# Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'VRAM: {vram / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
!pip install -q transformers av
!git clone https://github.com/AirKyzzZ/pkvision.git
%cd pkvision

In [ ]:
# ============================================
# PICK ONE METHOD — run only the one you want
# ============================================

# --- OPTION A: Google Drive (FAST — recommended for large files) ---
# 1. Drag training_data.zip into drive.google.com
# 2. Uncomment and run these 3 lines:

from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/training_data.zip /content/pkvision/ && cd /content/pkvision && unzip -q -o training_data.zip && echo "Done! Training data ready."

# --- OPTION B: Direct upload (SLOW — ok for small files < 1GB) ---
# Uncomment these lines instead:

# from google.colab import files
# print('Upload training_data.zip:')
# uploaded = files.upload()
# !unzip -q -o training_data.zip
# print('Done!')

In [ ]:
# Load manifest
import json
from pathlib import Path

with open('data/training/manifest.json') as f:
    manifest = json.load(f)

classes = manifest['classes']
print(f'Classes: {len(classes)}')
for cls, count in manifest['class_counts'].items():
    bar = '#' * min(count, 40)
    print(f'  {cls:25s} {count:3d} {bar}')
print(f'\nTotal: {len(manifest["samples"])} samples')

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor

class PkVisionDataset(Dataset):
    def __init__(self, frames_dir, manifest, processor, num_frames=16):
        self.frames_dir = Path(frames_dir)
        self.processor = processor
        self.num_frames = num_frames
        self.classes = manifest['classes']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = [(s['file'], self.class_to_idx[s['class']]) for s in manifest['samples'] if s['class'] in self.class_to_idx]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fn, label = self.samples[idx]
        frames = np.load(self.frames_dir / fn)
        T = frames.shape[0]
        indices = np.linspace(0, T-1, min(self.num_frames, T), dtype=int).tolist()
        while len(indices) < self.num_frames: indices.append(indices[-1])
        sampled = [frames[i] for i in indices[:self.num_frames]]
        inputs = self.processor(sampled, return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), label

# Config
EPOCHS = 20
BATCH_SIZE = 8  # Increase to 16 on A100
LR = 5e-5

In [ ]:
# Load model
model_name = 'MCG-NJU/videomae-base-finetuned-kinetics'
processor = VideoMAEImageProcessor.from_pretrained(model_name)
model = VideoMAEForVideoClassification.from_pretrained(model_name)

num_classes = len(classes)
hidden_size = model.classifier.in_features
model.classifier = nn.Linear(hidden_size, num_classes)

# Freeze backbone (only train 8K params)
for param in model.videomae.parameters():
    param.requires_grad = False

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Training {trainable:,} / {sum(p.numel() for p in model.parameters()):,} params')

In [ ]:
# Dataset + split
dataset = PkVisionDataset('data/training/frames', manifest, processor)
val_size = max(1, int(len(dataset) * 0.15))
train_ds, val_ds = random_split(dataset, [len(dataset)-val_size, val_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# Train!
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    tl, tc, tt = 0., 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(pixel_values=bx)
        loss = criterion(out.logits, by)
        loss.backward()
        optimizer.step()
        tl += loss.item()*bx.size(0); tc += (out.logits.argmax(1)==by).sum().item(); tt += bx.size(0)
    scheduler.step()

    model.eval()
    vl, vc, vt = 0., 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            out = model(pixel_values=bx)
            loss = criterion(out.logits, by)
            vl += loss.item()*bx.size(0); vc += (out.logits.argmax(1)==by).sum().item(); vt += bx.size(0)

    ta, va = tc/max(tt,1), vc/max(vt,1)
    history['train_loss'].append(tl/max(tt,1)); history['val_loss'].append(vl/max(vt,1))
    history['train_acc'].append(ta); history['val_acc'].append(va)
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | train {tl/max(tt,1):.4f} {ta:.0%} | val {vl/max(vt,1):.4f} {va:.0%} | {time.time()-t0:.1f}s')

    if va >= best_val_acc:
        best_val_acc = va
        torch.save({'model_state_dict': model.state_dict(), 'classes': classes, 'epoch': epoch, 'val_acc': va,
                     'config': {'model_name': model_name, 'num_classes': num_classes, 'hidden_size': hidden_size}}, 'pkvision_videomae.pt')

print(f'\nBest val accuracy: {best_val_acc:.0%}')

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot([a*100 for a in history['train_acc']], label='Train'); ax2.plot([a*100 for a in history['val_acc']], label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
# Test on a video
import av
from google.colab import files as colab_files

print('Upload a test video:')
uploaded = colab_files.upload()
test_video = list(uploaded.keys())[0]

c = av.open(test_video)
af = [f.to_ndarray(format='rgb24') for f in c.decode(video=0)]; c.close()
indices = np.linspace(0, len(af)-1, 16, dtype=int)
frames = [af[i] for i in indices]

ckpt = torch.load('pkvision_videomae.pt', map_location=device)
m = VideoMAEForVideoClassification.from_pretrained(model_name)
m.classifier = nn.Linear(hidden_size, num_classes)
m.load_state_dict(ckpt['model_state_dict']); m.to(device); m.eval()

inp = processor(frames, return_tensors='pt'); inp = {k:v.to(device) for k,v in inp.items()}
with torch.no_grad(): probs = torch.softmax(m(pixel_values=inp['pixel_values']).logits, dim=1)[0]

si = probs.argsort(descending=True)
print(f'\nResults for {test_video}:')
print('-'*45)
for r, idx in enumerate(si):
    cls = classes[idx.item()]; cf = probs[idx].item()
    print(f'{r+1}. {cls:25s} {cf:5.1%} {"#"*int(cf*30)}{" << TOP" if r==0 else ""}')
print(f'\nDETECTED: {classes[si[0].item()].upper().replace("_"," ")} ({probs[si[0]].item():.0%})')

In [ ]:
# Download trained model
colab_files.download('pkvision_videomae.pt')
print('Put this file in: pkvision/data/models/pkvision_videomae.pt')
print('Then test locally: python scripts/test_model.py --input your_video.mp4')